# 🌊 CalCOFI Oceanographic Data Exploration

This notebook explores the **CalCOFI (California Cooperative Oceanic Fisheries Investigations)** dataset, one of the longest-running and most important oceanographic time-series in the world.

We will analyze two primary datasets:
1. **Bottle Data**: Physical and chemical measurements at specific depths.
2. **Cast Data**: Metadata for each deployment (time, location, weather conditions).

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Premium styling
sns.set_theme(style='darkgrid', palette='viridis')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

## 📂 Data Loading

We'll load both datasets. The Bottle data is significantly larger, so we'll load columns of interest to save memory.

In [ ]:
# File Paths
BOTTLE_FILE = '194903-202105_Bottle.csv'
CAST_FILE = '194903-202105_Cast.csv'

print("Loading Cast data...")
df_cast = pd.read_csv(CAST_FILE, low_memory=False)

print("Loading Bottle data (key columns only)... ")
cols_bottle = ['Cst_Cnt', 'Depthm', 'T_degC', 'Salnty', 'O2ml_L']
df_bottle = pd.read_csv(BOTTLE_FILE, usecols=cols_bottle, low_memory=False)

print(f"Loaded {len(df_cast)} casts and {len(df_bottle)} bottle measurements.")

## 🧪 Data Preprocessing

Oceanographic data often contains missing values or extreme outliers. Let's clean the data and merge the two sets.

In [ ]:
# Remove rows with missing Temp or Salinity for physical analysis
df_phys = df_bottle.dropna(subset=['T_degC', 'Salnty'])

# Merge with Cast data to get spatial/temporal info
df_merged = pd.merge(
    df_phys, 
    df_cast[['Cst_Cnt', 'Lat_Dec', 'Lon_Dec', 'Date', 'Year', 'Month']], 
    on='Cst_Cnt'
)

df_merged['Date'] = pd.to_datetime(df_merged['Date'])
df_merged.head()

## 🌡️ Temperature vs. Depth

The thermocline is the layer in the ocean where temperature decreases rapidly with depth.

In [ ]:
plt.figure(figsize=(10, 8))
sns.scatterplot(data=df_merged.sample(10000), x='T_degC', y='Depthm', alpha=0.3, s=10)
plt.gca().invert_yaxis() # Depth is usually plotted negative or with inverted axis
plt.title('Temperature vs. Depth Profile (Sampleed 10k points)')
plt.xlabel('Temperature (°C)')
plt.ylabel('Depth (meters)')
plt.show()

## 🧂 T-S Diagram (Temperature vs. Salinity)

The T-S diagram is a fundamental tool in oceanography used to identify water masses. Water masses have unique combinations of temperature and salinity.

In [ ]:
fig = px.scatter(
    df_merged.sample(15000), 
    x='Salnty', 
    y='T_degC', 
    color='Depthm', 
    hover_data=['Date'],
    title='T-S Diagram (Temperature vs. Salinity)',
    labels={'Salnty': 'Salinity (PSU)', 'T_degC': 'Temperature (°C)'},
    color_continuous_scale=px.colors.sequential.Deep
)
fig.update_layout(template='plotly_dark')
fig.show()

## 🗺️ Geospatial Distribution of Casts

Where were these measurements taken?

In [ ]:
# Aggregating unique cast locations
df_locations = df_cast.dropna(subset=['Lat_Dec', 'Lon_Dec']).drop_duplicates(subset=['Lat_Dec', 'Lon_Dec'])

fig_map = px.scatter_geo(
    df_locations.sample(min(len(df_locations), 5000)), 
    lat='Lat_Dec', 
    lon='Lon_Dec', 
    hover_name='Sta_ID',
    title='CalCOFI Cast Locations (Subsampled)',
    scope='usa',
    center={'lat': 34, 'lon': -120}
)
fig_map.update_geos(projection_type="natural earth")
fig_map.show()

## 📈 Temporal Trends

How has the average surface temperature changed over the years?

In [ ]:
# Filter for surface measurements (depth < 10m)
df_surface = df_merged[df_merged['Depthm'] < 10]
yearly_avg = df_surface.groupby('Year')['T_degC'].mean().reset_index()

plt.figure(figsize=(14, 6))
sns.lineplot(data=yearly_avg, x='Year', y='T_degC', marker='o', color='#ff6b6b')
plt.title('Average Surface Temperature Trend (1949 - 2021)')
plt.ylabel('Average Temperature (°C)')
plt.show()